# Visualise Imputed Data

Same heatmaps as `visualise.py`, but built from the imputed CSVs in each dataset's `imputation.output_dir`. Outputs are written to `<image_dir>_imputed/` so the originals aren't overwritten.

In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from tqdm import tqdm

from visualise import generate_comparison_df


def plot_site_comparison_heatmap_ordered(df, feature_name, limit_dict, image_dir, column_order):
    """Like visualise.plot_site_comparison_heatmap, but uses an externally
    supplied column order instead of sorting by NaN count."""
    cols = [c for c in column_order if c in df.columns]
    extras = [c for c in df.columns if c not in cols]  # any new sites get appended
    df = df.reindex(columns=cols + extras)
    df_transposed = df.T

    fig, ax = plt.subplots(figsize=(20, len(df.columns) * 0.025))
    vmax = limit_dict.get(feature_name, df.max().max())
    sns.heatmap(
        df_transposed,
        cmap="YlOrRd",
        cbar_kws={"label": feature_name},
        xticklabels=False,
        yticklabels=False,
        ax=ax,
        mask=df_transposed.isna(),
        vmin=0,
        vmax=vmax,
    )

    tick_dates = pd.date_range(start=df.index[0], end=df.index[-1], freq="6MS")
    tick_positions = [df.index.get_loc(ts) for ts in tick_dates if ts in df.index]
    tick_labels = [ts.strftime("%b %Y") for ts in tick_dates if ts in df.index]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=45, fontsize=8)

    plt.title(f"{feature_name} Across Sites Over Time", fontsize=16, pad=20)
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Site", fontsize=12)
    plt.tight_layout()

    safe = feature_name.split(" ")[0]
    plt.savefig(os.path.join(image_dir, f"{safe}.png"), dpi=500, bbox_inches="tight")
    plt.close()


def original_column_order(original_dicts_dir, feature):
    """Reproduce the column ordering used by visualise.plot_site_comparison_heatmap
    on the original (un-imputed) dict: ascending NaN count."""
    safe = feature.split(" ")[0]
    path = os.path.join(original_dicts_dir, f"{safe}_df.csv")
    if not os.path.isfile(path):
        return None
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    return df.isnull().sum().sort_values().index.tolist()

In [2]:
CONFIG_PATH = "config.yaml"
MAKE_DICTS = True  # rebuild per-pollutant comparison DataFrames from imputed CSVs

with open(CONFIG_PATH) as f:
    all_cfg = yaml.safe_load(f)

# Pick which datasets to visualise. By default: every dataset that has both
# a `visualise` and an `imputation` block.
datasets = [k for k, v in all_cfg.items() if isinstance(v, dict) and "visualise" in v and "imputation" in v]
datasets

['epa', 'cpcb', 'aurn', 'eea_fr', 'eea_de', 'sinaica', 'cnemc']

In [3]:
def build_imputed_cfg(dataset_key):
    """Merge a dataset's `visualise` + `imputation` config into a vis-style cfg
    pointing at the imputed CSVs, with separate output dirs."""
    vis = all_cfg[dataset_key]["visualise"]
    imp = all_cfg[dataset_key]["imputation"]

    image_dir = vis["output"]["image_dir"].rstrip("/") + "_imputed"
    dicts_dir = vis["output"]["dicts_dir"].rstrip("/") + "_imputed"

    return {
        "folder": imp["output_dir"],
        "date_range": imp.get("date_range", vis["date_range"]),
        "features": imp.get("features", vis["features"]),
        "pollutant_limits": vis["pollutant_limits"],
        "dicts_dir": dicts_dir,
        "image_dir": image_dir,
        "original_dicts_dir": vis["output"]["dicts_dir"],
    }

imputed_cfgs = {k: build_imputed_cfg(k) for k in datasets}
for k, c in imputed_cfgs.items():
    print(f"{k}: folder={c['folder']}\n     -> images={c['image_dir']}, dicts={c['dicts_dir']}")

epa: folder=/home/rishi/ML Projects/Air Pollution/EPA/imputed
     -> images=visualize_epa_images_imputed, dicts=home/rishi/ML Projects/Air Pollution/EPA/visualize_dicts_imputed
cpcb: folder=/home/rishi/ML Projects/Air Pollution/CPCB/sites_imputed
     -> images=visualize_cpcb_images_imputed, dicts=/home/rishi/ML Projects/Air Pollution/CPCB/vis_dicts_imputed
aurn: folder=/home/rishi/ML Projects/Air Pollution/AURN/imputed
     -> images=visualize_aurn_images_imputed, dicts=/home/rishi/ML Projects/Air Pollution/AURN/vis_dicts_imputed
eea_fr: folder=/home/rishi/ML Projects/Air Pollution/EEA/FR/imputed
     -> images=visualize_eea_fr_images_imputed, dicts=/home/rishi/ML Projects/Air Pollution/EEA/FR/vis_dicts_imputed
eea_de: folder=/home/rishi/ML Projects/Air Pollution/EEA/DE/imputed
     -> images=visualize_eea_de_images_imputed, dicts=/home/rishi/ML Projects/Air Pollution/EEA/DE/vis_dicts_imputed
sinaica: folder=/home/rishi/ML Projects/Air Pollution/SINAICA/imputed
     -> images=visuali

In [4]:
for dataset_key, cfg in imputed_cfgs.items():
    print(f"\n=== {dataset_key} ===")
    if not os.path.isdir(cfg["folder"]):
        print(f"  skipping, imputed folder not found: {cfg['folder']}")
        continue

    os.makedirs(cfg["dicts_dir"], exist_ok=True)
    os.makedirs(cfg["image_dir"], exist_ok=True)

    full_index = pd.date_range(cfg["date_range"]["start"], cfg["date_range"]["end"], freq="h")

    if MAKE_DICTS:
        print("  building per-pollutant comparison DataFrames...")
        for feature in tqdm(cfg["features"]):
            safe = feature.split(" ")[0]
            feature_df = generate_comparison_df(feature, cfg["folder"], full_index)
            feature_df.to_csv(os.path.join(cfg["dicts_dir"], f"{safe}_df.csv"))

    print("  plotting heatmaps...")
    for feature in tqdm(cfg["features"]):
        safe = feature.split(" ")[0]
        feature_df = pd.read_csv(
            os.path.join(cfg["dicts_dir"], f"{safe}_df.csv"),
            index_col=0,
            parse_dates=True,
        )
        col_order = original_column_order(cfg["original_dicts_dir"], feature)
        if col_order is None:
            print(f"    no original dict for {feature}, falling back to NaN-sort order")
            col_order = feature_df.isnull().sum().sort_values().index.tolist()
        plot_site_comparison_heatmap_ordered(
            feature_df, feature, cfg["pollutant_limits"], cfg["image_dir"], col_order
        )

    print(f"  done -> {cfg['image_dir']}/")


=== epa ===
  building per-pollutant comparison DataFrames...


100%|██████████| 6/6 [00:33<00:00,  5.66s/it]


  plotting heatmaps...


100%|██████████| 6/6 [00:46<00:00,  7.80s/it]


  done -> visualize_epa_images_imputed/

=== cpcb ===
  building per-pollutant comparison DataFrames...


...

  done -> visualize_sinaica_images_imputed/

=== cnemc ===
  building per-pollutant comparison DataFrames...


100%|██████████| 6/6 [02:55<00:00, 29.27s/it]


  plotting heatmaps...


100%|██████████| 6/6 [04:00<00:00, 40.09s/it]

  done -> visualize_cnemc_images_imputed/
